# Phase 5: Policy Evaluation

**Goal:** find out whether ranking customers by estimated uplift, `tau(X) = m1(X) - m0(X)`, finds more incremental purchasers than ranking them by purchase likelihood, `m1(X)`, and how certain that is.

The notebook needs the Parquet files from notebooks 01 to 03, including the frozen score files from Phase 4. The rules were committed in `reports/decisions.md`, under "Rules for Phase 5", before any targeting result existed.

- **Stage 2, development:** out-of-fold rows only. Arm balance at other budgets, sanity checks, the strategy comparison, the disagreement analysis and the RFM cut points. No holdout label or holdout score is loaded.
- **Stage 3, holdout:** added after the lock commit, behind `RUN_HOLDOUT_EVALUATION`.

`target` means a purchase in an undocumented period after communication, and `treatment_flg` records SMS communication status. Observed checks are consistent with random assignment, but randomisation is not documented, so causal readings depend on that assumption.

## Setup

In [1]:
import hashlib
import os
import re
import sys
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklift.metrics import qini_auc_score, qini_curve, uplift_auc_score, uplift_curve
from sklift.metrics import uplift_at_k as sklift_uplift_at_k

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

sys.path.insert(0, str(ROOT / "notebooks"))
from targeting import (
    arm_balance,
    assert_exact_tau,
    overall_difference,
    score_ranks,
    select_by_rank,
    select_top_k,
    stratified_resample_indices,
    treated_share_band,
    uplift_at_budgets,
    uplift_at_k_by_rank,
)

# scikit-uplift 0.5.1 calls a scikit-learn helper that is deprecated in this version. The result is unaffected.
warnings.filterwarnings("ignore", message="Function stable_cumsum is deprecated", category=FutureWarning)

TABLES = Path("outputs/tables")
FIGURES = Path("outputs/figures")
PROCESSED = Path("data/processed")
FEATURES_PATH = PROCESSED / "customer_features.parquet"
SPLITS_PATH = PROCESSED / "splits.parquet"
LABELS_PATH = PROCESSED / "uplift_train.parquet"
OOF_SCORES_PATH = PROCESSED / "scores_oof.parquet"
DECISIONS_PATH = Path("reports/decisions.md")

SEED = 42
BOOTSTRAP_RESAMPLES = 2_000
SHUFFLES = 1_000
SHUFFLE_SE_LIMIT = 4
SKLIFT_TOLERANCE = 1e-12
BUDGETS = (5, 10, 20, 30)
OTHER_BALANCE_BUDGETS = (5, 10, 30)
PRIMARY_BUDGET = 20
Z = stats.norm.ppf(0.975)

EXPECTED_DEVELOPMENT = 160_031
EXPECTED_TREATED_DEVELOPMENT = 79_985
EXPECTED_CONTROL_DEVELOPMENT = 80_046
EXPECTED_HOLDOUT = 40_008
EXPECTED_LABELLED = 200_039

RFM_COLUMNS = {"recency": "days_since_last_purchase", "frequency": "txn_count_hist", "monetary": "spend_hist"}
RFM_SHARE_LIMITS = (0.20, 0.50)
RECENCY_NAMES = {1: "Recent", 2: "Moderately recent", 3: "Least recent"}

# Holdout guard. Set to True only after the project owner confirms the lock commit, and put that hash here.
RUN_HOLDOUT_EVALUATION = False
LOCK_COMMIT_SHA = None

# Chart colours: categorical slots 1 and 2 of the reference palette, with neutral ink for the random line and text.
COLOURS = {"Response": "#2a78d6", "Uplift": "#eb6834", "Random": "#8a8983", "ink": "#3d3d3a", "grid": "#e4e3de"}
plt.rcParams.update({
    "axes.edgecolor": COLOURS["grid"], "axes.labelcolor": COLOURS["ink"], "xtick.color": COLOURS["ink"],
    "ytick.color": COLOURS["ink"], "axes.grid": True, "grid.color": COLOURS["grid"], "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "lines.linewidth": 2,
})


def show(table, formats):
    """Display a copy of the table with the given columns formatted. jinja2 is not installed, so pandas styling is not used."""
    shown = table.copy()
    for column, pattern in formats.items():
        if column in shown:
            shown[column] = shown[column].map(pattern.format)
    display(shown)

## 1. Helpers used only in this notebook

The rules shared with other notebooks live in `notebooks/targeting.py`, including Uplift@K and the overall difference. These helpers follow the Phase 5 rules directly: the unpooled Wald interval, the percentile interval and the RFM tertiles, including the rank-based fallback.

In [2]:
def wald_difference(treatment, target):
    """Rates, difference and unpooled Wald 95% interval, using the Phase 1 method."""
    treated_rate, control_rate = target[treatment == 1].mean(), target[treatment == 0].mean()
    n_treated, n_control = (treatment == 1).sum(), (treatment == 0).sum()
    se = np.sqrt(treated_rate * (1 - treated_rate) / n_treated + control_rate * (1 - control_rate) / n_control)
    difference = treated_rate - control_rate
    return {"n_treated": int(n_treated), "n_control": int(n_control), "treated_rate": treated_rate, "control_rate": control_rate,
            "difference": difference, "ci_low": difference - Z * se, "ci_high": difference + Z * se}


def percentile_interval(samples):
    low, high = np.percentile(samples, [2.5, 97.5])
    return low, high


def value_tertiles(development_values, values):
    """Tertile 1 if at most the first cut point, 2 if at most the second, otherwise 3. Cut points from development rows."""
    cuts = np.quantile(development_values, [1 / 3, 2 / 3])
    return 1 + (values > cuts[0]).astype(int) + (values > cuts[1]).astype(int), cuts


def rank_tertiles(development_values, development_ids, values, ids):
    """The logged fallback: development rows sorted by value, then client_id; rank r goes to tertile floor(3r / n) + 1.

    The cut points are the (value, client_id) pairs of the last development customer in tertiles 1 and 2, and every
    customer is placed by comparing their own (value, client_id) with those pairs.
    """
    development_ids = np.asarray(development_ids, dtype=str)
    ids = np.asarray(ids, dtype=str)
    order = np.lexsort((development_ids, development_values))
    n = len(order)
    tertile_in_order = 3 * np.arange(n) // n + 1
    pairs = []
    for tertile in (1, 2):
        last = order[np.flatnonzero(tertile_in_order == tertile)[-1]]
        pairs.append((development_values[last], development_ids[last]))

    def at_most(pair):
        value, client_id = pair
        return (values < value) | ((values == value) & (ids <= client_id))

    tertiles = 1 + (~at_most(pairs[0])).astype(int) + (~at_most(pairs[1])).astype(int)
    return tertiles, pairs


def rfm_segment(recency_tertile, frequency_tertile, monetary_tertile):
    """Segments 1 to 6. Higher frequency and monetary value means a mean tertile above 2; exactly 2 counts as lower."""
    higher = (frequency_tertile + monetary_tertile) > 4
    return 2 * (recency_tertile - 1) + 1 + higher.astype(int)

## 2. Known-answer checks

The shared Phase 5 functions and the helpers above run on small constructed inputs first. Every result must match a value worked out by hand. The Phase 4 checks on the older shared functions still run in `03_response_vs_uplift.ipynb`. A few are repeated here because `select_by_rank` was refactored in Phase 5.

In [3]:
def rejected(call):
    try:
        call()
    except ValueError:
        return True
    return False


# Unchanged Phase 4 behaviour after the refactor: sizes, tie order and repeated ranks.
for customers_in_group, percent, size in ((160_031, 20, 32_006), (40_008, 20, 8_002), (10, 25, 3), (30, 5, 2), (50, 5, 3), (3, 100, 3)):
    assert select_by_rank(np.arange(customers_in_group), percent).sum() == size
assert select_top_k(["d", "b", "a", "c"], [0.5, 0.5, 0.5, 0.5], 50).tolist() == [False, True, True, False]
assert select_by_rank(score_ranks(["a", "b", "c", "d"], [0.4, 0.3, 0.2, 0.1])[[3, 0, 0, 1, 2]], 40).tolist() == [False, True, True, False, False]
assert rejected(lambda: select_by_rank([-1, 1, 2], 34)) and rejected(lambda: select_by_rank([0, 1, 2], 0.2))

# Group sizes quoted in the rules.
for n, sizes in ((40_008, (2_000, 4_001, 8_002, 12_002)), (160_031, (8_002, 16_003, 32_006, 48_009))):
    assert tuple(select_by_rank(np.arange(n), budget).sum() for budget in BUDGETS) == sizes
assert select_by_rank(np.arange(160_031), 50).sum() == 80_016

# Uplift@K. Top 4 of 10: treated rows 0 and 2 both buy (1.0), control rows 1 and 3 give 0.5, so uplift is 0.5.
RANKS = np.arange(10)
ARM = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])
BUY = np.array([1, 0, 1, 1, 0, 0, 1, 0, 0, 0])
assert uplift_at_k_by_rank(RANKS, ARM, BUY, 40) == 0.5
assert uplift_at_k_by_rank(RANKS, ARM, BUY, 20) == 1.0
assert np.isclose(uplift_at_k_by_rank(RANKS, ARM, BUY, 100), 0.6 - 0.2)
shuffled_order = np.array([9, 3, 0, 7, 1, 8, 2, 6, 4, 5])
assert uplift_at_k_by_rank(RANKS[shuffled_order], ARM[shuffled_order], BUY[shuffled_order], 40) == 0.5
assert rejected(lambda: uplift_at_k_by_rank(RANKS, ARM, BUY, 10))  # one treated row, no control row
assert uplift_at_k_by_rank(np.array([0, 0, 1, 5]), np.array([1, 1, 0, 0]), np.array([1, 1, 0, 1]), 75) == 1.0
for bad_call in (
    lambda: uplift_at_k_by_rank(RANKS, ARM[:-1], BUY, 40),
    lambda: uplift_at_k_by_rank(RANKS, ARM * 2, BUY, 40),
    lambda: uplift_at_k_by_rank(RANKS, ARM, BUY + 1, 40),
    lambda: uplift_at_k_by_rank(RANKS, ARM.astype(float) + 0.5, BUY, 40),
    lambda: uplift_at_k_by_rank(RANKS, ARM, BUY, 0.4),
):
    assert rejected(bad_call)

# Several budgets from one sort give exactly the select_by_rank groups, including repeated ranks.
check_rng = np.random.default_rng(0)
for _ in range(300):
    n = int(check_rng.integers(20, 400))
    ranks, arm, buy = check_rng.integers(0, n, n), check_rng.integers(0, 2, n), check_rng.integers(0, 2, n)
    valid, invalid = [], []
    for percent in (5, 10, 20, 30, 50, 100):
        group = select_by_rank(ranks, percent)
        (invalid if arm[group].min() == arm[group].max() else valid).append(percent)
    together = uplift_at_budgets(ranks, arm, buy, valid)
    assert all(together[percent] == uplift_at_k_by_rank(ranks, arm, buy, percent) for percent in valid)
    assert all(rejected(lambda: uplift_at_budgets(ranks, arm, buy, (percent,))) for percent in invalid)
assert rejected(lambda: uplift_at_budgets(RANKS, ARM, BUY, ()))

# The scikit-uplift match: -ranks has no ties, and a whole-number k avoids its rounding down.
assert sklift_uplift_at_k(BUY, -RANKS, ARM, strategy="overall", k=4) == uplift_at_k_by_rank(RANKS, ARM, BUY, 40)

# Stratified resamples: one at a time, arm counts kept, indices stay in their arm, reproducible from the seed.
arm = np.array([1, 1, 1, 0, 0, 1, 0, 1, 1, 0])
first, second = list(stratified_resample_indices(arm, 5, 42)), list(stratified_resample_indices(arm, 5, 42))
assert len(first) == 5 and all(np.array_equal(a, b) for a, b in zip(first, second))
assert not np.array_equal(first[0], first[1])
assert all((arm[rows] == 1).sum() == 6 and (arm[rows] == 0).sum() == 4 and len(rows) == 10 for rows in first)
assert not isinstance(stratified_resample_indices(arm, 3, 42), (list, tuple, np.ndarray))
for bad_call in (
    lambda: list(stratified_resample_indices([1, 1, 1], 2, 42)),
    lambda: list(stratified_resample_indices([1, 0, 2], 2, 42)),
    lambda: list(stratified_resample_indices(arm, 0, 42)),
    lambda: list(stratified_resample_indices(arm, 2.0, 42)),
):
    assert rejected(bad_call)

# Treated-share band: 3.5 x sqrt(0.25 / size), matching the logged table.
assert round(treated_share_band(2_000) * 100, 2) == 3.91 and round(treated_share_band(4_001) * 100, 2) == 2.77
assert round(treated_share_band(12_002) * 100, 2) == 1.60 and round(treated_share_band(48_009) * 100, 2) == 0.80
assert all(rejected(lambda: treated_share_band(bad)) for bad in (0, -5, 10.0, True))

# Overall difference and the Wald interval. Treated 1 of 2, control 0 of 2: SE = sqrt(0.25 / 2).
known = wald_difference(np.array([1, 1, 0, 0]), np.array([1, 0, 0, 0]))
assert known["difference"] == 0.5 and np.isclose(known["ci_high"] - 0.5, Z * np.sqrt(0.125))
assert overall_difference(np.array([1, 1, 0, 0]), np.array([1, 0, 0, 0])) == 0.5
assert overall_difference(ARM, BUY) == uplift_at_k_by_rank(RANKS, ARM, BUY, 100)
assert all(rejected(bad) for bad in (lambda: overall_difference(ARM, BUY + 1), lambda: overall_difference(np.ones(10, int), BUY),
                                     lambda: overall_difference(ARM, np.where(BUY == 1, np.nan, 0.0))))

# Seeds and budgets are validated, so resamples stay reproducible and results cannot misalign.
assert all(rejected(lambda: stratified_resample_indices(arm, 2, bad)) for bad in (None, 4.2, True))
assert all(rejected(lambda: uplift_at_budgets(RANKS, ARM, BUY, bad)) for bad in ((20, 20, 40), (p for p in (20, 40)), 20))

# Value tertiles: 1 to 9 cut at 3.667 and 6.333. With ties, every 0 shares tertile 1 and tertile 2 is empty.
tertiles, cuts = value_tertiles(np.arange(1, 10), np.arange(1, 10))
assert tertiles.tolist() == [1, 1, 1, 2, 2, 2, 3, 3, 3] and np.allclose(cuts, [11 / 3, 19 / 3])
tied = np.array([0, 0, 0, 0, 1, 2])
assert value_tertiles(tied, tied)[0].tolist() == [1, 1, 1, 1, 3, 3]

# Rank-based fallback: a and b go to 1, c and d to 2, e and f to 3; cut pairs (0, "b") and (0, "d").
tied_ids = np.array(["a", "b", "c", "d", "e", "f"])
tertiles, pairs = rank_tertiles(tied, tied_ids, tied, tied_ids)
assert tertiles.tolist() == [1, 1, 2, 2, 3, 3] and pairs == [(0, "b"), (0, "d")]
others, _ = rank_tertiles(tied, tied_ids, np.array([0, 0, 1.5, -1]), np.array(["a0", "bb", "z", "zz"]))
assert others.tolist() == [1, 2, 3, 1]
# Input order does not matter.
reversed_rows = np.arange(6)[::-1]
assert rank_tertiles(tied[reversed_rows], tied_ids[reversed_rows], tied, tied_ids)[0].tolist() == [1, 1, 2, 2, 3, 3]

# Segments: a mean F and M tertile of exactly 2 counts as lower.
assert rfm_segment(np.array([1, 1, 3, 3, 2]), np.array([3, 3, 1, 3, 2]), np.array([1, 2, 1, 3, 3])).tolist() == [1, 2, 5, 6, 4]

print("All known-answer checks passed")

All known-answer checks passed


## 3. Load the development customers

One query reads the out-of-fold scores, the development labels and the three RFM columns, filtered to `split = 'development'`. A second reads the three RFM columns for all 200,039 labelled customers, with no labels, so the development cut points can be applied to everyone. No holdout score or holdout label is loaded in Stage 2.

In [4]:
con = duckdb.connect()


def query(sql):
    return con.execute(sql).fetchdf()


# The saved split must be the one recorded in Phase 1.
splits = query(f"SELECT client_id, split, fold FROM '{SPLITS_PATH}' ORDER BY client_id")
canonical = "".join(
    f"{client},{split},{'' if pd.isna(fold) else int(fold)}\n"
    for client, split, fold in zip(splits["client_id"], splits["split"], splits["fold"])
)
recorded_split_hash = re.search(r"\*\*Split SHA-256:\*\* `([0-9a-f]{64})`", DECISIONS_PATH.read_text(encoding="utf-8")).group(1)
assert hashlib.sha256(canonical.encode("utf-8")).hexdigest() == recorded_split_hash, "The split differs from the Phase 1 record"
holdout_ids = set(splits.loc[splits["split"] == "holdout", "client_id"])
assert len(holdout_ids) == EXPECTED_HOLDOUT and len(splits) == EXPECTED_LABELLED

rfm_select = ", ".join(f"f.{column}" for column in RFM_COLUMNS.values())
development = query(
    f"""
    SELECT o.client_id, o.fold AS score_fold, s.fold, o.m1_score, o.m0_score, o.tau, l.treatment_flg, l.target, {rfm_select}
    FROM '{OOF_SCORES_PATH}' AS o
    JOIN '{SPLITS_PATH}' AS s USING (client_id)
    JOIN '{LABELS_PATH}' AS l USING (client_id)
    JOIN '{FEATURES_PATH}' AS f USING (client_id)
    WHERE s.split = 'development'
    ORDER BY o.client_id
    """
)
everyone = query(
    f"""
    SELECT s.client_id, s.split, {rfm_select}
    FROM '{SPLITS_PATH}' AS s
    JOIN '{FEATURES_PATH}' AS f USING (client_id)
    ORDER BY s.client_id
    """
)
con.close()

assert len(development) == EXPECTED_DEVELOPMENT
assert development["client_id"].is_unique and development["client_id"].is_monotonic_increasing
assert holdout_ids.isdisjoint(development["client_id"])
assert (development["score_fold"] == development["fold"]).all(), "Score folds differ from the split folds"
assert (development["treatment_flg"] == 1).sum() == EXPECTED_TREATED_DEVELOPMENT
assert (development["treatment_flg"] == 0).sum() == EXPECTED_CONTROL_DEVELOPMENT
assert development["target"].isin([0, 1]).all()
assert len(everyone) == EXPECTED_LABELLED and everyone["client_id"].is_unique
assert set(everyone.loc[everyone["split"] == "holdout", "client_id"]) == holdout_ids

ids = development["client_id"].to_numpy(dtype=object)
m1 = development["m1_score"].to_numpy()
m0 = development["m0_score"].to_numpy()
tau = development["tau"].to_numpy()
treatment = development["treatment_flg"].to_numpy(dtype=np.int64)
target = development["target"].to_numpy(dtype=np.int64)
fold_of_row = development["fold"].to_numpy(dtype=int)
assert_exact_tau(m1, m0, tau)
del splits, canonical

print(f"{len(development):,} development customers: {treatment.sum():,} treated, {(1 - treatment).sum():,} control")
print(f"Overall development difference: {overall_difference(treatment, target) * 100:+.2f} points")

160,031 development customers: 79,985 treated, 80,046 control
Overall development difference: +3.32 points


## 4. Arm balance at other budgets

The Phase 4 balance check covered the top 20%. For `m1` and `tau` at 5%, 10% and 30%, the treated share must be within 3.5 binomial standard errors of 50% at that group size. This uses `treatment_flg` but not `target`. The table is written before the stop rule is applied, so a stop still leaves its evidence.

In [5]:
SCORES = {"Response": m1, "Uplift": tau}
RANKS_BY_STRATEGY = {name: score_ranks(ids, score) for name, score in SCORES.items()}

balance_rows = []
for name, ranks in RANKS_BY_STRATEGY.items():
    for budget in OTHER_BALANCE_BUDGETS:
        group = select_by_rank(ranks, budget)
        share = treatment[group].mean()
        band = treated_share_band(int(group.sum()))
        balance_rows.append({"strategy": name, "score": "m1" if name == "Response" else "tau", "budget_pct": budget,
                             "group_size": int(group.sum()), "treated_share": share, "band": band,
                             "within_band": abs(share - 0.5) <= band})
balance_table = pd.DataFrame(balance_rows)
balance_table.to_csv(TABLES / "oof_treated_share_bands.csv", index=False)
show(balance_table, {"treated_share": "{:.2%}", "band": "±{:.2%}"})

outside = balance_table[~balance_table["within_band"]]
assert outside.empty, f"Stop and investigate: treated share outside its band\n{outside}"

,strategy,score,budget_pct,group_size,treated_share,band,within_band
0,Response,m1,5,8002,50.34%,±1.96%,True
1,Response,m1,10,16003,50.18%,±1.38%,True
2,Response,m1,30,48009,50.06%,±0.80%,True
3,Uplift,tau,5,8002,49.99%,±1.96%,True
4,Uplift,tau,10,16003,49.89%,±1.38%,True
5,Uplift,tau,30,48009,49.64%,±0.80%,True


## 5. Sanity checks

**Shuffle check:** at each budget, 1,000 random orderings, from a new generator seeded 42, replace the scores. Their mean Uplift@K must sit within 4 Monte Carlo standard errors of the overall difference. This tests the selection and Uplift@K code, not the models.

**Match with scikit-uplift:** for the response and uplift strategies at each budget, scikit-uplift's `uplift_at_k` with `-ranks` and the whole-number group size must equal the custom value within 1e-12.

In [6]:
def shuffle_check(treatment, target, seed):
    rng = np.random.default_rng(seed)
    n = len(treatment)
    values = np.empty((SHUFFLES, len(BUDGETS)))
    for row in range(SHUFFLES):
        # A random permutation of 0..n-1 is a random set of ranks, passed to select_by_rank through Uplift@K.
        ranks = rng.permutation(n)
        values[row] = [uplift_at_k_by_rank(ranks, treatment, target, budget) for budget in BUDGETS]
    overall = overall_difference(treatment, target)
    table = pd.DataFrame({"budget_pct": BUDGETS, "overall_difference": overall, "shuffle_mean": values.mean(axis=0),
                          "shuffle_sd": values.std(axis=0, ddof=1)})
    table["monte_carlo_se"] = table["shuffle_sd"] / np.sqrt(SHUFFLES)
    table["standard_errors_away"] = (table["shuffle_mean"] - overall) / table["monte_carlo_se"]
    table["passes"] = table["standard_errors_away"].abs() <= SHUFFLE_SE_LIMIT
    return table


def sklift_check(ranks_by_strategy, treatment, target):
    rows = []
    for name, ranks in ranks_by_strategy.items():
        for budget in BUDGETS:
            size = int(select_by_rank(ranks, budget).sum())
            ours = uplift_at_k_by_rank(ranks, treatment, target, budget)
            theirs = sklift_uplift_at_k(target, -ranks, treatment, strategy="overall", k=size)
            rows.append({"strategy": name, "budget_pct": budget, "group_size": size, "custom": ours, "scikit_uplift": theirs,
                         "absolute_difference": abs(ours - theirs), "passes": abs(ours - theirs) <= SKLIFT_TOLERANCE})
    return pd.DataFrame(rows)


oof_shuffle = shuffle_check(treatment, target, SEED)
oof_shuffle.to_csv(TABLES / "oof_shuffle_check.csv", index=False)
display(oof_shuffle)

oof_sklift = sklift_check(RANKS_BY_STRATEGY, treatment, target)
oof_sklift.to_csv(TABLES / "oof_sklift_match.csv", index=False)
display(oof_sklift)

assert oof_shuffle["passes"].all(), "Stop and investigate: the shuffle mean is too far from the overall difference"
assert oof_sklift["passes"].all(), "Stop and investigate: the custom Uplift@K differs from scikit-uplift"

,budget_pct,overall_difference,shuffle_mean,shuffle_sd,monte_carlo_se,standard_errors_away,passes
0,5,0.033229,0.033108,0.010738,0.000340,-0.356617,True
1,10,0.033229,0.033252,0.007358,0.000233,0.100660,True
2,20,0.033229,0.033264,0.004844,0.000153,0.230144,True
3,30,0.033229,0.033234,0.003649,0.000115,0.047791,True


,strategy,budget_pct,group_size,custom,scikit_uplift,absolute_difference,passes
0,Response,5,8002,0.003105,0.003105,0.0,True
1,Response,10,16003,0.013306,0.013306,0.0,True
2,Response,20,32006,0.014504,0.014504,0.0,True
3,Response,30,48009,0.018359,0.018359,0.0,True
4,Uplift,5,8002,0.115716,0.115716,0.0,True
5,Uplift,10,16003,0.092828,0.092828,0.0,True
6,Uplift,20,32006,0.071214,0.071214,0.0,True
7,Uplift,30,48009,0.058727,0.058727,0.0,True


## 6. Strategy comparison on out-of-fold rows

The pooled ranking of all 160,031 out-of-fold rows is the main development view. Every interval comes from the same 2,000 stratified resamples, drawn one at a time from a new generator seeded 42. Each resample is used for every strategy, budget and metric before the next is drawn, so every difference is paired.

- **Random** is the expected value of random targeting: the overall difference on the same rows. Its interval leaves out the noise of a single random draw.
- **Contact everyone** is the same difference at 100%.
- On out-of-fold rows every result is **Secondary**, or **Exploratory** at 5%. Only the holdout comparison at 20% is primary.

The intervals capture uncertainty from the evaluation sample, not from refitting the models. This cell takes several minutes.

In [ ]:
def evaluate(client_ids, m1_scores, tau_scores, treatment, target, seed):
    """Point estimates and paired bootstrap draws for every strategy, budget and metric on one evaluation set."""
    scores = {"Response": m1_scores, "Uplift": tau_scores}
    ranks = {name: score_ranks(client_ids, score) for name, score in scores.items()}
    point = {
        "overall": overall_difference(treatment, target),
        "uplift": {name: uplift_at_budgets(ranks[name], treatment, target, BUDGETS) for name in scores},
        "qini": {name: qini_auc_score(target, score, treatment) for name, score in scores.items()},
        "auuc": {name: uplift_auc_score(target, score, treatment) for name, score in scores.items()},
    }
    draws = {
        "overall": np.empty(BOOTSTRAP_RESAMPLES),
        "uplift": {name: np.empty((BOOTSTRAP_RESAMPLES, len(BUDGETS))) for name in scores},
        "qini": {name: np.empty(BOOTSTRAP_RESAMPLES) for name in scores},
        "auuc": {name: np.empty(BOOTSTRAP_RESAMPLES) for name in scores},
    }
    for row, indices in enumerate(stratified_resample_indices(treatment, BOOTSTRAP_RESAMPLES, seed)):
        resampled_treatment, resampled_target = treatment[indices], target[indices]
        draws["overall"][row] = overall_difference(resampled_treatment, resampled_target)
        for name, score in scores.items():
            result = uplift_at_budgets(ranks[name][indices], resampled_treatment, resampled_target, BUDGETS)
            draws["uplift"][name][row] = [result[budget] for budget in BUDGETS]
            draws["qini"][name][row] = qini_auc_score(resampled_target, score[indices], resampled_treatment)
            draws["auuc"][name][row] = uplift_auc_score(resampled_target, score[indices], resampled_treatment)
    return {"point": point, "draws": draws, "ranks": ranks}


def result_label(evaluation_set, comparison, budget):
    if evaluation_set == "holdout" and comparison == "Uplift minus Response" and budget == PRIMARY_BUDGET:
        return "Primary"
    return "Exploratory" if budget == 5 else "Secondary"


def strategy_values(evaluation):
    """Point value and draws of Uplift@K for every strategy and budget, with Random as the overall difference."""
    point, draws = evaluation["point"], evaluation["draws"]
    values = {}
    for column, budget in enumerate(BUDGETS):
        values[("Random", budget)] = (point["overall"], draws["overall"])
        for name in ("Response", "Uplift"):
            values[(name, budget)] = (point["uplift"][name][budget], draws["uplift"][name][:, column])
    return values


def strategy_table(evaluation, evaluation_set):
    rows = []
    values = strategy_values(evaluation)
    entries = [(name, budget) for budget in BUDGETS for name in ("Random", "Response", "Uplift")]
    for name, budget in entries + [("Contact everyone", 100)]:
        value, samples = values[("Random", BUDGETS[0])] if name == "Contact everyone" else values[(name, budget)]
        low, high = percentile_interval(samples)
        per_100k = 100_000 * budget / 100
        rows.append({"set": evaluation_set, "strategy": "Random (expected value of random targeting)" if name == "Random" else name,
                     "budget_pct": budget, "uplift_pts": value * 100, "ci_low_pts": low * 100, "ci_high_pts": high * 100,
                     "per_100k": value * per_100k, "per_100k_ci_low": low * per_100k, "per_100k_ci_high": high * per_100k,
                     "label": "Exploratory" if budget == 5 else "Secondary"})
    return pd.DataFrame(rows)


def paired_table(evaluation, evaluation_set):
    rows = []
    values = strategy_values(evaluation)
    for budget in BUDGETS:
        for first, second in (("Uplift", "Response"), ("Uplift", "Random"), ("Response", "Random")):
            comparison = f"{first} minus {second}"
            value = values[(first, budget)][0] - values[(second, budget)][0]
            samples = values[(first, budget)][1] - values[(second, budget)][1]
            low, high = percentile_interval(samples)
            per_100k = 100_000 * budget / 100
            rows.append({"set": evaluation_set, "comparison": comparison, "budget_pct": budget,
                         "difference_pts": value * 100, "ci_low_pts": low * 100, "ci_high_pts": high * 100,
                         "per_100k": value * per_100k, "per_100k_ci_low": low * per_100k, "per_100k_ci_high": high * per_100k,
                         "label": result_label(evaluation_set, comparison, budget)})
    return pd.DataFrame(rows)


def ranking_table(evaluation, evaluation_set):
    rows = []
    point, draws = evaluation["point"], evaluation["draws"]
    for metric, title in (("qini", "Qini coefficient"), ("auuc", "AUUC")):
        for name in ("Response", "Uplift"):
            low, high = percentile_interval(draws[metric][name])
            rows.append({"set": evaluation_set, "metric": title, "strategy": name, "value": point[metric][name],
                         "ci_low": low, "ci_high": high, "label": "Secondary"})
        low, high = percentile_interval(draws[metric]["Uplift"] - draws[metric]["Response"])
        rows.append({"set": evaluation_set, "metric": title, "strategy": "Uplift minus Response",
                     "value": point[metric]["Uplift"] - point[metric]["Response"], "ci_low": low, "ci_high": high, "label": "Secondary"})
    return pd.DataFrame(rows)


oof_evaluation = evaluate(ids, m1, tau, treatment, target, SEED)
for name in SCORES:
    assert np.array_equal(oof_evaluation["ranks"][name], RANKS_BY_STRATEGY[name])
# The reported values come from the several-budget helper, so they must also match scikit-uplift.
for check in oof_sklift.itertuples():
    reported = oof_evaluation["point"]["uplift"][check.strategy][check.budget_pct]
    assert abs(reported - check.scikit_uplift) <= SKLIFT_TOLERANCE, (check.strategy, check.budget_pct)

oof_strategies = strategy_table(oof_evaluation, "out-of-fold")
oof_paired = paired_table(oof_evaluation, "out-of-fold")
oof_ranking = ranking_table(oof_evaluation, "out-of-fold")
oof_strategies.to_csv(TABLES / "oof_strategy_results.csv", index=False)
oof_paired.to_csv(TABLES / "oof_paired_differences.csv", index=False)
oof_ranking.to_csv(TABLES / "oof_ranking_metrics.csv", index=False)

points = {column: "{:+.2f}" for column in ("uplift_pts", "ci_low_pts", "ci_high_pts", "difference_pts")}
counts = {column: "{:+,.0f}" for column in ("per_100k", "per_100k_ci_low", "per_100k_ci_high")}
show(oof_strategies, {**{k: v for k, v in points.items() if k in oof_strategies}, **counts})
show(oof_paired, {**{k: v for k, v in points.items() if k in oof_paired}, **counts})
show(oof_ranking, {"value": "{:+.4f}", "ci_low": "{:+.4f}", "ci_high": "{:+.4f}"})

### By fold

Report only, with no interval and no stop rule. Each fold's own rows are ranked with `select_top_k`, and Uplift@20 is calculated within that fold.

In [ ]:
fold_rows = []
for fold in sorted(np.unique(fold_of_row)):
    rows = fold_of_row == fold
    values = {name: uplift_at_k_by_rank(score_ranks(ids[rows], score[rows]), treatment[rows], target[rows], PRIMARY_BUDGET)
              for name, score in SCORES.items()}
    fold_rows.append({"fold": fold, "customers": int(rows.sum()), "response_uplift_at_20_pts": values["Response"] * 100,
                      "uplift_uplift_at_20_pts": values["Uplift"] * 100,
                      "uplift_minus_response_pts": (values["Uplift"] - values["Response"]) * 100})
fold_table = pd.DataFrame(fold_rows)
fold_table.to_csv(TABLES / "oof_by_fold.csv", index=False)
show(fold_table, {column: "{:+.2f}" for column in fold_table.columns if column.endswith("_pts")})

### Qini and uplift curves

Drawn with Matplotlib from `qini_curve` and `uplift_curve` in `sklift.metrics`, because `sklift.viz` does not import with the installed scikit-learn. The dashed line joins the start and end of each curve, which is what random ranking gives on average. The dotted line marks the 20% budget.

In [ ]:
def plot_curves(scores, treatment, target, title, path):
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for axis, (curve, label) in zip(axes, ((qini_curve, "Qini curve: incremental purchasers"), (uplift_curve, "Uplift curve"))):
        for name, score in scores.items():
            x, y = curve(target, score, treatment)
            share = x / len(target) * 100
            axis.plot(share, y, color=COLOURS[name], label=f"{name} ({'m1' if name == 'Response' else 'tau'})")
        axis.plot([0, 100], [0, y[-1]], color=COLOURS["Random"], linestyle="--", linewidth=1.5, label="Random")
        axis.axvline(PRIMARY_BUDGET, color=COLOURS["ink"], linestyle=":", linewidth=1)
        axis.set_xlim(0, 100)
        axis.set_xlabel("Customers targeted, ranked by score (%)")
        axis.set_title(label, color=COLOURS["ink"], loc="left", fontsize=10)
    axes[0].legend(frameon=False, loc="upper left")
    figure.suptitle(title, color=COLOURS["ink"], x=0.01, ha="left", fontsize=11)
    figure.tight_layout()
    figure.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


plot_curves(SCORES, treatment, target, "Out-of-fold development rows (160,031 customers)", FIGURES / "oof_qini_uplift_curves.png")

## 7. Disagreement analysis

Pooled out-of-fold rows only. "Higher" is the top 50% of each score from `select_top_k`. These are model-defined groups. They do not identify individual persuadable customers, and the observed uplift in each is **Measured** with an unpooled Wald interval.

In [ ]:
higher_likelihood = select_top_k(ids, m1, 50)
higher_uplift = select_top_k(ids, tau, 50)
QUADRANTS = [
    ("Likely buyers, more apparent SMS effect", higher_likelihood & higher_uplift),
    ("Likely buyers, less apparent SMS effect", higher_likelihood & ~higher_uplift),
    ("Possible persuadables", ~higher_likelihood & higher_uplift),
    ("Less likely buyers, less apparent SMS effect", ~higher_likelihood & ~higher_uplift),
]
assert sum(group.sum() for _, group in QUADRANTS) == EXPECTED_DEVELOPMENT

quadrant_rows, profile_rows = [], []
for name, group in QUADRANTS:
    result = wald_difference(treatment[group], target[group])
    quadrant_rows.append({"quadrant": name, "customers": int(group.sum()), **result})
    profile = {"quadrant": name}
    for label, column in RFM_COLUMNS.items():
        profile[f"{column}_mean"] = development.loc[group, column].mean()
        profile[f"{column}_median"] = development.loc[group, column].median()
    profile_rows.append(profile)
quadrant_table = pd.DataFrame(quadrant_rows)
profile_table = pd.DataFrame(profile_rows)

overlap_rows = []
for budget in BUDGETS:
    uplift_list = select_by_rank(RANKS_BY_STRATEGY["Uplift"], budget)
    response_list = select_by_rank(RANKS_BY_STRATEGY["Response"], budget)
    assert uplift_list.sum() == response_list.sum()
    overlap_rows.append({"budget_pct": budget, "list_size": int(uplift_list.sum()), "in_both": int((uplift_list & response_list).sum()),
                         "share_of_uplift_list_in_response_list": (uplift_list & response_list).sum() / uplift_list.sum()})
overlap_table = pd.DataFrame(overlap_rows)

quadrant_table.to_csv(TABLES / "disagreement_quadrants.csv", index=False)
profile_table.to_csv(TABLES / "disagreement_profile.csv", index=False)
overlap_table.to_csv(TABLES / "list_overlap.csv", index=False)
show(quadrant_table, {"treated_rate": "{:.2%}", "control_rate": "{:.2%}", "difference": "{:+.2%}", "ci_low": "{:+.2%}", "ci_high": "{:+.2%}"})
display(profile_table.round(2))
show(overlap_table, {"share_of_uplift_list_in_response_list": "{:.1%}"})

In [ ]:
figure, axis = plt.subplots(figsize=(8, 3.2))
positions = np.arange(len(quadrant_table))[::-1]
axis.errorbar(quadrant_table["difference"] * 100, positions,
              xerr=[(quadrant_table["difference"] - quadrant_table["ci_low"]) * 100, (quadrant_table["ci_high"] - quadrant_table["difference"]) * 100],
              fmt="o", markersize=8, color=COLOURS["Uplift"], ecolor=COLOURS["Uplift"], elinewidth=2, capsize=0)
axis.axvline(overall_difference(treatment, target) * 100, color=COLOURS["Random"], linestyle="--", linewidth=1.5)
axis.axvline(0, color=COLOURS["ink"], linewidth=0.8)
axis.set_yticks(positions, [f"{name}  (n = {count:,})" for name, count in zip(quadrant_table["quadrant"], quadrant_table["customers"])])
axis.set_xlabel("Observed treated minus control purchase rate (points), with 95% interval")
axis.set_title("Disagreement quadrants, out-of-fold rows. Dashed line: overall difference", color=COLOURS["ink"], loc="left", fontsize=10)
axis.grid(axis="y", visible=False)
figure.tight_layout()
figure.savefig(FIGURES / "disagreement_quadrants.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. RFM cut points

The cut points use the 160,031 development rows and no outcomes. If any tertile of a variable holds under 20% or over 50% of development rows, that variable uses the logged rank-based fallback. The segments are then applied to all 200,039 customers. Segment effects are calculated once, in Stage 3.

In [ ]:
for column in RFM_COLUMNS.values():
    assert development[column].notna().all() and everyone[column].notna().all(), f"Stop: {column} has nulls"

dev_rows = everyone["split"].to_numpy() == "development"
assert np.array_equal(everyone.loc[dev_rows, "client_id"].to_numpy(), development["client_id"].to_numpy())

tertiles_all, cut_rows, share_rows = {}, [], []
for label, column in RFM_COLUMNS.items():
    dev_values = development[column].to_numpy(dtype=float)
    all_values = everyone[column].to_numpy(dtype=float)
    tertiles, cuts = value_tertiles(dev_values, all_values)
    shares = np.bincount(tertiles[dev_rows], minlength=4)[1:] / dev_rows.sum()
    for tertile, share in enumerate(shares, start=1):
        share_rows.append({"variable": label, "column": column, "method": "value", "tertile": tertile, "development_share": share})
    method = "value"
    if (shares < RFM_SHARE_LIMITS[0]).any() or (shares > RFM_SHARE_LIMITS[1]).any():
        method = "rank fallback"
        tertiles, pairs = rank_tertiles(dev_values, development["client_id"], all_values, everyone["client_id"])
        shares = np.bincount(tertiles[dev_rows], minlength=4)[1:] / dev_rows.sum()
        for tertile, share in enumerate(shares, start=1):
            share_rows.append({"variable": label, "column": column, "method": method, "tertile": tertile, "development_share": share})
        cut_rows.append({"variable": label, "column": column, "method": method, "value_cut_1": cuts[0], "value_cut_2": cuts[1],
                         "cut_1": f"({pairs[0][0]:g}, {pairs[0][1]})", "cut_2": f"({pairs[1][0]:g}, {pairs[1][1]})"})
    else:
        cut_rows.append({"variable": label, "column": column, "method": method, "value_cut_1": cuts[0], "value_cut_2": cuts[1],
                         "cut_1": f"{cuts[0]:g}", "cut_2": f"{cuts[1]:g}"})
    tertiles_all[label] = tertiles

cut_table = pd.DataFrame(cut_rows)
share_table = pd.DataFrame(share_rows)
segments = rfm_segment(tertiles_all["recency"], tertiles_all["frequency"], tertiles_all["monetary"])
everyone["rfm_segment"] = segments

segment_rows = []
for segment in range(1, 7):
    in_segment = segments == segment
    segment_rows.append({"segment": segment, "recency": RECENCY_NAMES[(segment + 1) // 2],
                         "frequency_and_monetary": "Higher" if segment % 2 == 0 else "Lower",
                         "development_customers": int((in_segment & dev_rows).sum()), "all_customers": int(in_segment.sum())})
segment_table = pd.DataFrame(segment_rows)
assert segment_table["all_customers"].sum() == EXPECTED_LABELLED

cut_table.to_csv(TABLES / "rfm_cut_points.csv", index=False)
share_table.to_csv(TABLES / "rfm_tertile_shares.csv", index=False)
segment_table.to_csv(TABLES / "rfm_segment_sizes.csv", index=False)
display(cut_table)
show(share_table, {"development_share": "{:.1%}"})
display(segment_table)

### Values quoted in `reports/decisions.md`

The development results and the lock record quote these values, so a rerun that changes any of them stops here.

In [ ]:
def pinned(table, **filters):
    rows = table
    for column, value in filters.items():
        rows = rows[rows[column] == value]
    assert len(rows) == 1, filters
    return rows.iloc[0]


primary_like = pinned(oof_paired, comparison="Uplift minus Response", budget_pct=20)
quoted = {
    "overall difference (pts)": (round(overall_difference(treatment, target) * 100, 2), 3.32),
    "Response Uplift@20 (pts)": (round(pinned(oof_strategies, strategy="Response", budget_pct=20)["uplift_pts"], 2), 1.45),
    "Uplift Uplift@20 (pts)": (round(pinned(oof_strategies, strategy="Uplift", budget_pct=20)["uplift_pts"], 2), 7.12),
    "Uplift minus Response at 20% (pts)": (round(primary_like["difference_pts"], 2), 5.67),
    "its interval low (pts)": (round(primary_like["ci_low_pts"], 2), 4.46),
    "its interval high (pts)": (round(primary_like["ci_high_pts"], 2), 6.92),
    "customers on both 20% lists": (int(pinned(overlap_table, budget_pct=20)["in_both"]), 1_193),
    "customers on both 5% lists": (int(pinned(overlap_table, budget_pct=5)["in_both"]), 0),
    "RFM methods": (tuple(cut_table["method"]), ("value", "value", "value")),
    "RFM cut points": (tuple(round(value, 2) for value in cut_table[["value_cut_1", "value_cut_2"]].to_numpy().ravel()),
                       (2.0, 7.0, 10.0, 22.0, 4045.09, 8973.04)),
    "segment sizes, all customers": (tuple(segment_table["all_customers"]), (27_843, 48_222, 36_640, 22_564, 53_259, 11_511)),
}


def rounded(table, column, digits=2):
    return tuple(round(float(value), digits) for value in table[column])


quoted.update({
    "treated shares, m1 then tau, 5/10/30% (%)": (tuple(round(value * 100, 2) for value in balance_table["treated_share"]),
                                                  (50.34, 50.18, 50.06, 49.99, 49.89, 49.64)),
    "largest shuffle gap (Monte Carlo SEs)": (round(float(oof_shuffle["standard_errors_away"].abs().max()), 2), 0.36),
    "Uplift@K, all strategies and budgets (pts)": (rounded(oof_strategies, "uplift_pts"),
        (3.32, 0.31, 11.57, 3.32, 1.33, 9.28, 3.32, 1.45, 7.12, 3.32, 1.84, 5.87, 3.32)),
    "paired differences (pts)": (rounded(oof_paired, "difference_pts"),
        (11.26, 8.25, -3.01, 7.95, 5.96, -1.99, 5.67, 3.8, -1.87, 4.04, 2.55, -1.49)),
    "paired interval lows (pts)": (rounded(oof_paired, "ci_low_pts"),
        (9.01, 6.22, -3.81, 6.35, 4.58, -2.71, 4.46, 2.88, -2.52, 2.95, 1.77, -2.05)),
    "paired interval highs (pts)": (rounded(oof_paired, "ci_high_pts"),
        (13.57, 10.4, -2.18, 9.62, 7.43, -1.27, 6.92, 4.78, -1.23, 4.99, 3.28, -0.85)),
    "Qini and AUUC values": (rounded(oof_ranking, "value", 4), (-0.0053, 0.0131, 0.0184, -0.0083, 0.0204, 0.0287)),
    "per-fold differences (pts)": (rounded(fold_table, "uplift_minus_response_pts"), (6.14, 3.34, 6.88, 5.49, 6.79)),
    "quadrant uplift (pts)": (tuple(round(float(value) * 100, 2) for value in quadrant_table["difference"]), (3.86, 1.59, 5.31, 1.79)),
    "quadrant sizes": (tuple(quadrant_table["customers"]), (34_523, 45_493, 45_493, 34_522)),
    "customers on both lists, 5/10/20/30%": (tuple(overlap_table["in_both"]), (0, 76, 1_193, 7_042)),
})
mismatches = {name: values for name, values in quoted.items() if values[0] != values[1]}
assert not mismatches, f"Stop: values differ from those quoted in the log: {mismatches}"
print("All quoted values match")

## Stage 2 conclusion

All values are **Measured** on the 160,031 out-of-fold development rows. They are **secondary**, or **exploratory** at 5%, and none of them is the primary result, which comes only from the holdout in Stage 3. Causal readings depend on the assignment assumption.

- **Checks:** every treated share is inside its band, the shuffle means sit within 0.4 Monte Carlo standard errors of the overall difference, and the custom Uplift@K matches scikit-uplift exactly.
- **Strategy comparison:** uplift targeting found more incremental purchasers than response targeting at every budget. At 20% the difference is +5.67 points, 95% interval +4.46 to +6.92, or about 1,134 per 100,000 customers. Response targeting fell below the expected value of random targeting at every budget.
- **By fold:** the 20% difference is positive in all five folds, from +3.34 to +6.88 points.
- **Disagreement:** the two rankings choose almost entirely different customers at small budgets. No customer is on both 5% lists, and 3.7% of the 20% uplift list is on the response list. These are model-defined groups, and no test between them was run.
- **RFM:** value-based cut points work for all three variables, so no fallback was needed.

**What this does not show:** every score here comes from a model that did not train on that customer, but these same rows chose the model setup in Phase 3, and five different model pairs produced the pooled ranking. Whether the difference holds on untouched customers is the Stage 3 question.

## Holdout evaluation (Stage 3)

Added after the lock commit. `RUN_HOLDOUT_EVALUATION` stays `False` until the project owner confirms that commit.